In [ ]:
# MarketMind Intelligence Platform V1
# Author: Sharique Mohammad
# Date: April 23, 2026

# MarketMind V1 - SEC Filings Analysis

**Notebook:** 04_sec_filings_analysis.ipynb  
**Purpose:** Analyze SEC filing patterns and timelines

---

## Overview

This notebook analyzes SEC filings from `gold.sec_filings`:
- Filing frequency and patterns
- Form type distribution (10-K, 10-Q, 8-K)
- Filing timelines by company
- Filing lag analysis
- Quarterly and annual patterns

**Data Period:** 2022-2026
**Companies:** AAPL, MSFT, GOOGL (+ others with ticker='UNKNOWN')
**Records:** 700 filings

In [ ]:
# Import libraries
import psycopg2
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Set style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('Set2')

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)

print('Libraries imported successfully!')

## 1. Database Connection & Data Loading

In [ ]:
# PostgreSQL connection
DB_CONFIG = {
    'host': '172.31.32.1',
    'port': 5432,
    'database': 'marketmind_v1',
    'user': 'postgres',
    'password': '0940'
}

# Load SEC filings data
conn = psycopg2.connect(**DB_CONFIG)

query = """
SELECT 
    ticker,
    company_name,
    form_type,
    filing_date,
    report_date,
    filing_year,
    filing_quarter,
    filing_category,
    is_periodic_report,
    filing_lag_days,
    accession_number
FROM gold.sec_filings
ORDER BY filing_date, ticker
"""

df = pd.read_sql(query, conn)
conn.close()

# Convert dates to datetime
df['filing_date'] = pd.to_datetime(df['filing_date'])
df['report_date'] = pd.to_datetime(df['report_date'])

print(f'Loaded {len(df):,} filings')
print(f'Date range: {df["filing_date"].min()} to {df["filing_date"].max()}')
print(f'Unique companies: {df["ticker"].nunique()}')
print('\nForm type breakdown:')
print(df['form_type'].value_counts())
df.head(10)

## 2. Filing Type Distribution

In [ ]:
# Count filings by form type
form_counts = df['form_type'].value_counts()

# Create pie chart and bar chart
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

# Pie chart
colors = ['steelblue', 'darkorange', 'green']
ax1.pie(form_counts.values, labels=form_counts.index, autopct='%1.1f%%', 
        colors=colors, startangle=90, textprops={'fontsize': 11, 'fontweight': 'bold'})
ax1.set_title('SEC Filing Distribution by Form Type', fontsize=12, fontweight='bold')

# Bar chart
bars = ax2.bar(form_counts.index, form_counts.values, color=colors, edgecolor='black', linewidth=1.5)
ax2.set_xlabel('Form Type', fontsize=12, fontweight='bold')
ax2.set_ylabel('Number of Filings', fontsize=12, fontweight='bold')
ax2.set_title('Filing Counts by Form Type', fontsize=12, fontweight='bold')
ax2.grid(axis='y', alpha=0.3)

# Add value labels on bars
for bar, val in zip(bars, form_counts.values):
    ax2.text(bar.get_x() + bar.get_width()/2., bar.get_height(),
            f'{val}',
            ha='center', va='bottom', fontsize=11, fontweight='bold')

plt.tight_layout()
plt.show()

print('Form Type Descriptions:')
print('  10-K: Annual Report (comprehensive financial statements)')
print('  10-Q: Quarterly Report (unaudited financial statements)')
print('  8-K: Current Report (material events, changes)')

## 3. Filing Timeline - All Companies

In [ ]:
# Group filings by month and form type
df['year_month'] = df['filing_date'].dt.to_period('M')
monthly_filings = df.groupby(['year_month', 'form_type']).size().reset_index(name='count')
monthly_filings['year_month'] = monthly_filings['year_month'].dt.to_timestamp()

# Plot timeline
fig, ax = plt.subplots(figsize=(14, 7))

for form_type in monthly_filings['form_type'].unique():
    data = monthly_filings[monthly_filings['form_type'] == form_type]
    ax.plot(data['year_month'], data['count'], marker='o', markersize=5, 
            label=form_type, linewidth=2.5)

ax.set_xlabel('Date', fontsize=12, fontweight='bold')
ax.set_ylabel('Number of Filings', fontsize=12, fontweight='bold')
ax.set_title('SEC Filing Timeline by Form Type (2022-2026)', fontsize=14, fontweight='bold')
ax.legend(loc='best', fontsize=11)
ax.grid(True, alpha=0.3)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 4. Filings by Company (Excluding UNKNOWN)

In [ ]:
# Filter out UNKNOWN tickers
known_tickers = df[df['ticker'] != 'UNKNOWN'].copy()

if len(known_tickers) > 0:
    # Count filings per company and form type
    company_filings = known_tickers.groupby(['ticker', 'form_type']).size().reset_index(name='count')
    pivot = company_filings.pivot(index='ticker', columns='form_type', values='count').fillna(0)
    
    # Plot stacked bar chart
    fig, ax = plt.subplots(figsize=(12, 6))
    pivot.plot(kind='bar', stacked=True, ax=ax, color=['steelblue', 'darkorange', 'green'], 
               edgecolor='black', linewidth=1.2)
    
    ax.set_xlabel('Company Ticker', fontsize=12, fontweight='bold')
    ax.set_ylabel('Number of Filings', fontsize=12, fontweight='bold')
    ax.set_title('SEC Filings by Company and Form Type', fontsize=14, fontweight='bold')
    ax.legend(title='Form Type', fontsize=10)
    ax.grid(axis='y', alpha=0.3)
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.show()
    
    print('\nFilings per company:')
    print(pivot.astype(int))
else:
    print('No filings with known tickers found in dataset')

## 5. Filing Lag Analysis

In [ ]:
# Analyze filing lag (days between report date and filing date)
lag_data = df[df['filing_lag_days'].notna()].copy()

if len(lag_data) > 0:
    # Plot distribution by form type
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))
    
    # Histogram
    for form_type in lag_data['form_type'].unique():
        data = lag_data[lag_data['form_type'] == form_type]['filing_lag_days']
        ax1.hist(data, bins=20, alpha=0.6, label=form_type, edgecolor='black')
    
    ax1.set_xlabel('Filing Lag (days)', fontsize=12, fontweight='bold')
    ax1.set_ylabel('Frequency', fontsize=12, fontweight='bold')
    ax1.set_title('Distribution of Filing Lags', fontsize=12, fontweight='bold')
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    
    # Box plot
    lag_data.boxplot(column='filing_lag_days', by='form_type', ax=ax2)
    ax2.set_xlabel('Form Type', fontsize=12, fontweight='bold')
    ax2.set_ylabel('Filing Lag (days)', fontsize=12, fontweight='bold')
    ax2.set_title('Filing Lag by Form Type', fontsize=12, fontweight='bold')
    ax2.get_figure().suptitle('')  # Remove default title
    
    plt.tight_layout()
    plt.show()
    
    print('Filing Lag Statistics by Form Type:')
    print('=' * 80)
    for form_type in lag_data['form_type'].unique():
        data = lag_data[lag_data['form_type'] == form_type]['filing_lag_days']
        print(f'{form_type}:')
        print(f'  Mean: {data.mean():.1f} days')
        print(f'  Median: {data.median():.1f} days')
        print(f'  Min: {data.min():.0f} days')
        print(f'  Max: {data.max():.0f} days')
        print()
else:
    print('No filing lag data available')

## 6. Quarterly Filing Patterns

In [ ]:
# Analyze quarterly patterns (10-Q filings)
quarterly = df[(df['form_type'] == '10-Q') & (df['filing_quarter'].notna())].copy()

if len(quarterly) > 0:
    # Count filings by quarter
    quarter_counts = quarterly['filing_quarter'].value_counts().sort_index()
    
    # Plot
    fig, ax = plt.subplots(figsize=(10, 6))
    bars = ax.bar(quarter_counts.index, quarter_counts.values, 
                   color='darkorange', edgecolor='black', linewidth=1.5)
    
    ax.set_xlabel('Quarter', fontsize=12, fontweight='bold')
    ax.set_ylabel('Number of 10-Q Filings', fontsize=12, fontweight='bold')
    ax.set_title('10-Q Filings by Quarter', fontsize=14, fontweight='bold')
    ax.set_xticks([1, 2, 3, 4])
    ax.set_xticklabels(['Q1', 'Q2', 'Q3', 'Q4'])
    ax.grid(axis='y', alpha=0.3)
    
    # Add value labels
    for bar, val in zip(bars, quarter_counts.values):
        ax.text(bar.get_x() + bar.get_width()/2., bar.get_height(),
                f'{val}',
                ha='center', va='bottom', fontsize=11, fontweight='bold')
    
    plt.tight_layout()
    plt.show()
    
    print('10-Q Filings per Quarter:')
    for q, count in quarter_counts.items():
        print(f'  Q{int(q)}: {count} filings')
else:
    print('No quarterly (10-Q) filings found')

## 7. Annual Filing Patterns

In [ ]:
# Analyze annual patterns (10-K filings)
annual = df[df['form_type'] == '10-K'].copy()

if len(annual) > 0:
    # Count filings by year
    year_counts = annual['filing_year'].value_counts().sort_index()
    
    # Plot
    fig, ax = plt.subplots(figsize=(10, 6))
    bars = ax.bar(year_counts.index, year_counts.values, 
                   color='steelblue', edgecolor='black', linewidth=1.5)
    
    ax.set_xlabel('Filing Year', fontsize=12, fontweight='bold')
    ax.set_ylabel('Number of 10-K Filings', fontsize=12, fontweight='bold')
    ax.set_title('10-K Filings by Year', fontsize=14, fontweight='bold')
    ax.grid(axis='y', alpha=0.3)
    
    # Add value labels
    for bar, val in zip(bars, year_counts.values):
        ax.text(bar.get_x() + bar.get_width()/2., bar.get_height(),
                f'{val}',
                ha='center', va='bottom', fontsize=11, fontweight='bold')
    
    plt.tight_layout()
    plt.show()
    
    print('10-K Filings per Year:')
    for year, count in year_counts.items():
        print(f'  {int(year)}: {count} filings')
else:
    print('No annual (10-K) filings found')

## 8. Filing Category Breakdown

In [ ]:
# Analyze filing categories
category_data = df[df['filing_category'].notna()].copy()

if len(category_data) > 0:
    category_counts = category_data['filing_category'].value_counts()
    
    # Plot
    fig, ax = plt.subplots(figsize=(12, 6))
    bars = ax.barh(range(len(category_counts)), category_counts.values, 
                    color='green', edgecolor='black', linewidth=1.2)
    
    ax.set_yticks(range(len(category_counts)))
    ax.set_yticklabels(category_counts.index)
    ax.set_xlabel('Number of Filings', fontsize=12, fontweight='bold')
    ax.set_title('SEC Filings by Category', fontsize=14, fontweight='bold')
    ax.grid(axis='x', alpha=0.3)
    
    # Add value labels
    for bar, val in zip(bars, category_counts.values):
        ax.text(val + 5, bar.get_y() + bar.get_height()/2.,
                f'{val}',
                ha='left', va='center', fontsize=10, fontweight='bold')
    
    plt.tight_layout()
    plt.show()
    
    print('Filing Categories:')
    for cat, count in category_counts.items():
        print(f'  {cat}: {count} filings')
else:
    print('No filing category data available')

## 9. Most Recent Filings

In [ ]:
# Show most recent 20 filings
recent = df.nlargest(20, 'filing_date')[['ticker', 'company_name', 'form_type', 
                                           'filing_date', 'filing_category']]

print('Most Recent 20 SEC Filings:')
print('=' * 100)
print(recent.to_string(index=False))
print('=' * 100)